# 02 — Silver SISMEPRE
## Transformación Bronze → Silver

Procesa las **7 tablas del modelo relacional SISMEPRE** (Sistema de Seguimiento de Metas del Impuesto Predial).

| Tabla | Descripción |
|-------|-------------|
| RENTAS_FORMULARIO | Instrumento de medición del IP |
| RENTAS_PREGUNTAS | Ítems y preguntas del formulario |
| RENTAS_RESPUESTAS | Respuestas reportadas por municipalidad |
| RENTAS_ESTADISTICA | Resumen estadístico por pregunta |
| RENTAS_ESAT_ESTADISTICA_ATM | Histórico de Administración Tributaria |
| RENTAS_ENTIDAD_ESTADO | Estado de registro de la entidad |
| RENTAS_ANO_APLICACION | Años de aplicación disponibles |

**Fuente:** `data/bronze/sismepre/{tabla}/batch_*.json`  
**Destino:** `data/silver/sismepre/{tabla}/` (Parquet por tabla)


In [1]:
import sys, time
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.paths import BRONZE, SILVER, str_path, ensure_dirs
from src.spark_utils import get_spark, write_parquet
from src.transformations.common import clean_ghost_nulls
from core.audit.control_manager import ControlManager, ExecutionStatus

print("Imports OK")

Imports OK


In [2]:
# ── Tablas SISMEPRE a procesar ─────────────────────────────────────────────────
SISMEPRE_TABLES = [
    "RENTAS_FORMULARIO",
    "RENTAS_PREGUNTAS",
    "RENTAS_RESPUESTAS",
    "RENTAS_ESTADISTICA",
    "RENTAS_ESAT_ESTADISTICA_ATM",
    "RENTAS_ENTIDAD_ESTADO",
    "RENTAS_ANO_APLICACION",
]

BRONZE_BASE = BRONZE["sismepre"]
SILVER_BASE = SILVER["sismepre"]

print(f"Bronze base: {BRONZE_BASE}")
print(f"Silver base: {SILVER_BASE}")
print(f"Tablas a procesar: {len(SISMEPRE_TABLES)}")

ensure_dirs()

Bronze base: /workspace/data/bronze/sismepre
Silver base: /workspace/data/silver/sismepre
Tablas a procesar: 7
[OK] Estructura de directorios verificada en /workspace/data


In [3]:
spark = get_spark(app_name="MEF_Silver_SISMEPRE", memory="4g")

[OK] SparkSession lista | version=3.5.0 | master=local[*]


## Procesamiento por Tabla

Cada tabla se procesa de manera independiente:
1. Leer JSON de Bronze
2. Limpiar ghost nulls
3. Normalizar strings (columnas `_NOMBRE` y `_DESC` → UPPER)
4. Escribir Parquet a Silver

In [4]:
control = ControlManager(pipeline_name="silver_sismepre")
execution = control.start(input_parameters={"tables": SISMEPRE_TABLES})

metrics = {}
start_total = time.time()

try:
    for table_name in SISMEPRE_TABLES:
        print(f"\n{'─'*60}")
        print(f"Procesando: {table_name}")

        bronze_path = str_path(BRONZE_BASE / table_name.lower())
        silver_path = str_path(SILVER_BASE / table_name.lower())

        # Verificar si existen datos
        bronze_dir = BRONZE_BASE / table_name.lower()
        if not bronze_dir.exists() or not any(bronze_dir.rglob("*.json")):
            print(f"   Sin datos Bronze en {bronze_dir} — saltando")
            metrics[table_name] = 0
            continue

        # Lectura — Spark infiere el schema automáticamente
        raw_df = spark.read.option("multiline", "false").json(bronze_path)
        n_raw = raw_df.count()
        print(f"  Registros Bronze: {n_raw:,}")

        # Limpieza de ghost nulls
        clean_df = clean_ghost_nulls(raw_df)

        # Escritura Parquet
        n_clean = write_parquet(clean_df, silver_path, mode="overwrite")
        metrics[table_name] = n_clean

    elapsed = time.time() - start_total
    control.end(
        status=ExecutionStatus.SUCCESS,
        output_summary={**metrics, "elapsed_sec": round(elapsed, 1)}
    )
    print(f"\nSilver SISMEPRE completado en {elapsed:.1f}s")

except Exception as e:
    control.log_error("SilverSismepreError", str(e))
    control.end(status=ExecutionStatus.FAILED, output_summary={"error": str(e)})
    raise

2026-06-19 06:29:57 | INFO     | mef_dw.audit.control_manager | [AUDIT] Ejecución iniciada | pipeline=silver_sismepre id=a1374719

────────────────────────────────────────────────────────────
Procesando: RENTAS_FORMULARIO
  Registros Bronze: 98
  ✅ 98 filas → /workspace/data/silver/sismepre/rentas_formulario

────────────────────────────────────────────────────────────
Procesando: RENTAS_PREGUNTAS
  Registros Bronze: 836
  ✅ 836 filas → /workspace/data/silver/sismepre/rentas_preguntas

────────────────────────────────────────────────────────────
Procesando: RENTAS_RESPUESTAS
  Registros Bronze: 174,210
  ✅ 174,210 filas → /workspace/data/silver/sismepre/rentas_respuestas

────────────────────────────────────────────────────────────
Procesando: RENTAS_ESTADISTICA
  Registros Bronze: 233
  ✅ 233 filas → /workspace/data/silver/sismepre/rentas_estadistica

────────────────────────────────────────────────────────────
Procesando: RENTAS_ESAT_ESTADISTICA_ATM
  Registros Bronze: 134,144
  ✅ 13

In [5]:
# ── Resumen final ──────────────────────────────────────────────────────────────
print("\nResumen Silver SISMEPRE:")
print(f"  {'Tabla':<35} {'Filas':>10}")
print("  " + "-" * 47)
for table, count in metrics.items():
    status = "" if count > 0 else ""
    print(f"  {status} {table:<33} {count:>10,}")
print(f"\n  Total filas Silver: {sum(metrics.values()):,}")


Resumen Silver SISMEPRE:
  Tabla                                    Filas
  -----------------------------------------------
   RENTAS_FORMULARIO                         98
   RENTAS_PREGUNTAS                         836
   RENTAS_RESPUESTAS                    174,210
   RENTAS_ESTADISTICA                       233
   RENTAS_ESAT_ESTADISTICA_ATM          134,144
   RENTAS_ENTIDAD_ESTADO                 19,037
   RENTAS_ANO_APLICACION                     26

  Total filas Silver: 328,584
